# GHALogs metadata loader

Execution-verified adaptation of the authors’ loader. Correct JSONL line iteration, resumable batched upserts, source checksum verification, explicit oversized-record accounting. Source inputs and collection contents are never deleted.

In [1]:
import os, sys, json, gzip, csv, re, random, hashlib, struct, zlib, io, tarfile, time
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import pymongo
from pymongo import ReplaceOne
from bson import BSON
from urllib.request import Request, urlopen
from urllib.error import HTTPError

# Set these environment variables to your own external input/work locations.
# Git tracks only notebooks, provenance, compact results and review templates.
BASE = Path(os.getenv('GHA_DATA_ROOT', '../external_data')).resolve()
OUT = Path(os.getenv('GHA_WORK_ROOT', '../external_work')).resolve()
REPO = Path(os.getenv('GHA_SOURCE_ROOT', str(BASE / 'gha-dataset'))).resolve()
MONGODB_URI = os.getenv('MONGODB_URI', 'mongodb://127.0.0.1:27018')
DB_NAME = os.getenv('GHA_DB_NAME', 'gha-scraper')
assert OUT != BASE and OUT != REPO, 'Outputs must be isolated from inputs/source'
for name in ('metadata_audit', 'java_failure_candidates', 'raw_log_evidence',
             'raw', 'normalized', 'windows', 'cache'):
    (OUT/name).mkdir(exist_ok=True, parents=True)
client = pymongo.MongoClient(MONGODB_URI, serverSelectionTimeoutMS=5000)
display(client.admin.command('ping'))
db = client[DB_NAME]
mongo_repositories = db['repositories']; mongo_runs = db['runs']
def save_json(path, obj):
    path.write_text(json.dumps(obj, indent=2, default=str, ensure_ascii=False), encoding='utf-8')
def save_csv(path, rows):
    pd.DataFrame(rows).to_csv(path, index=False)
def jsonl(path, rows):
    with path.open('w', encoding='utf-8') as fd:
        for row in rows: fd.write(json.dumps(row, default=str, ensure_ascii=False)+'\n')
def get_json(url):
    with urlopen(Request(url, headers={'User-Agent':'GHALogs-publication-exploration'}), timeout=60) as response:
        return json.load(response)
print('Python', sys.version.split()[0], 'MongoDB', client.server_info()['version'])
print('External inputs:', BASE, 'External execution outputs:', OUT)

{'ok': 1.0}

Python 3.12.5 MongoDB 7.0.30
External inputs: D:\SJSU MSDA\DATA298A External execution outputs: D:\SJSU MSDA\DATA298A\ghalogs_publication_work\execution


In [2]:
record=get_json('https://zenodo.org/api/records/10154920')
files={f['key']:f for f in record['files']}
checks=[]
for name in ('repositories.json.gz','runs.json.gz'):
    path=BASE/name
    h=hashlib.md5()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''): h.update(chunk)
    expected=files[name]['checksum']
    actual='md5:'+h.hexdigest()
    assert path.stat().st_size==files[name]['size'] and actual==expected, (name,actual,expected)
    checks.append({'file':name,'bytes':path.stat().st_size,'checksum':actual,'verified':True})
save_json(OUT/'metadata_audit'/'metadata_input_checksums.json',checks)
display(pd.DataFrame(checks))
def import_jsonl(name, coll):
    state=db['_notebook_imports'].find_one({'_id':name})
    if state and state.get('complete') and coll.count_documents({})==state.get('imported',state.get('count')):
        print('Verified completed import:',name,state); return
    existing={r['_id'] for r in coll.find({}, {'_id':1})}
    n=0; batch=[]; oversized=[]
    with gzip.open(BASE/name,'rt',encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            document=json.loads(line)
            n+=1
            if document['_id'] in existing: continue
            bson_size=len(BSON.encode(document))
            if bson_size>16*1024*1024-1024:
                oversized.append({'source_run_id':document['_id'],'source_file':name,'jsonl_record_number':n,'bson_bytes':bson_size,
                    'reason':'MongoDB 16 MB document limit; raw source record preserved'})
                continue
            batch.append(ReplaceOne({'_id':document['_id']},document,upsert=True))
            if len(batch)>=1000:
                coll.bulk_write(batch,ordered=False); batch=[]
                if n%50000==0: print(name,n,flush=True)
    if batch: coll.bulk_write(batch,ordered=False)
    imported=coll.count_documents({})
    assert imported+len(oversized)==n, (imported,len(oversized),n)
    save_json(OUT/'metadata_audit'/(name+'.oversized.json'),oversized)
    db['_notebook_imports'].replace_one({'_id':name},{'_id':name,'complete':True,'source_records':n,'imported':imported,'oversized':len(oversized)},upsert=True)
    print('Imported',name,imported,'of',n,'oversized',len(oversized))
import_jsonl('repositories.json.gz',mongo_repositories)
import_jsonl('runs.json.gz',mongo_runs)
mongo_repositories.create_index('selected')
mongo_runs.create_index('repository_name')
mongo_runs.create_index([('repository_name',1),('workflow_path',1)])
# The upstream insight/archive indexes are not required by its exists/nonempty filters.
# Avoid indexing large nested arrays: retain query semantics without excessive index storage.

,file,bytes,checksum,verified
0,repositories.json.gz,69208463,md5:a44ed2c635348ecc8641f966748e32d2,True
1,runs.json.gz,1063390519,md5:d6c83b48b14351b16766ca1f01290e87,True


Verified completed import: repositories.json.gz {'_id': 'repositories.json.gz', 'complete': True, 'count': 240781}


Verified completed import: runs.json.gz {'_id': 'runs.json.gz', 'complete': True, 'source_records': 580641, 'imported': 580640, 'oversized': 1}


'repository_name_1_workflow_path_1'

In [3]:
display({'repositories':mongo_repositories.count_documents({}),'runs':mongo_runs.count_documents({}),'import_state':list(db['_notebook_imports'].find())})

{'repositories': 240781,
 'runs': 580640,
 'import_state': [{'_id': 'repositories.json.gz',
   'complete': True,
   'count': 240781},
  {'_id': 'runs.json.gz',
   'complete': True,
   'source_records': 580641,
   'imported': 580640,
   'oversized': 1}]}